In [ ]:
# STEP 1 — MOUNT GOOGLE DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


STEP 2 — IMPORT LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout

Step 3: Load Dataset

In [ ]:
file_path = "/content/drive/MyDrive/ai&ml_assessment/10.True vs. Fake News Dataset-20260508T165314Z-3-001/10.True vs. Fake News Dataset/truevsfakenews.csv"
df = pd.read_csv(file_path)
print("Rows:", len(df))
print("Columns:", list(df.columns))
print("First row:")
print(df.head(1))

Rows: 20000
Columns: ['text', 'label']
First row:
                                                text label
0  WASHINGTON (Reuters) - The Republican and Demo...  true


Step 4: Convert Labels to Numbers (true=1, fake=0)

In [ ]:
df['label_num'] = df['label'].map({'true': 1, 'fake': 0})
print("Number of true (1):", sum(df['label_num'] == 1))
print("Number of fake (0):", sum(df['label_num'] == 0))

Number of true (1): 10000
Number of fake (0): 10000


Step 5: Creating a small subset (2500 true, 2500 fake)for fast training

In [ ]:
true_df = df[df['label_num'] == 1]
fake_df = df[df['label_num'] == 0]

sample_true = true_df.sample(2500, random_state=42)
sample_fake = fake_df.sample(2500, random_state=42)

df_sample = pd.concat([sample_true, sample_fake])
df_sample = df_sample.sample(frac=1, random_state=42).reset_index(drop=True)

print("Subset size:", len(df_sample))
print("True in subset:", sum(df_sample['label_num'] == 1))
print("Fake in subset:", sum(df_sample['label_num'] == 0))

Subset size: 5000
True in subset: 2500
Fake in subset: 2500


Step 6: Cleaning text(Lowercase, remove Punctuation, Numbers)

In [ ]:
def clean_text(t):
    t = t.lower()
    t = t.translate(str.maketrans('', '', string.punctuation))
    t = re.sub(r'\d+', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df_sample['clean_text'] = df_sample['text'].apply(clean_text)
print("Original:", df_sample['text'].iloc[0][:80])
print("Cleaned:", df_sample['clean_text'].iloc[0][:80])

Original: NEW YORK (Reuters) - In the weeks since Hillary Clinton’s shock election defeat 
Cleaned: new york reuters in the weeks since hillary clinton’s shock election defeat to u


Step 7: Tokenization and Padding

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(df_sample['clean_text'])

sequences = tokenizer.texts_to_sequences(df_sample['clean_text'])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
y = df_sample['label_num'].values

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (5000, 100)
y shape: (5000,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

Train size: 4000
Test size: 1000


Step 8: Split into Training (80%) and Testing (20%)

In [ ]:
model_rnn = Sequential([
    Embedding(VOCAB_SIZE, 64, input_length=MAX_LEN),
    SimpleRNN(64),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model_rnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_rnn.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Step 9: Build Simple RNN Model

In [ ]:
history_rnn = model_rnn.fit(X_train, y_train, epochs=5, batch_size=32, validation_split=0.2, verbose=1)
loss_rnn, acc_rnn = model_rnn.evaluate(X_test, y_test)
print("RNN Test Accuracy:", acc_rnn)

Epoch 1/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 51ms/step - accuracy: 0.8175 - loss: 0.4141 - val_accuracy: 0.9588 - val_loss: 0.1176
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - accuracy: 0.9578 - loss: 0.1423 - val_accuracy: 0.9663 - val_loss: 0.0968
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.9550 - loss: 0.1358 - val_accuracy: 0.9425 - val_loss: 0.1630
Epoch 4/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.9856 - loss: 0.0556 - val_accuracy: 0.9787 - val_loss: 0.0841
Epoch 5/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 51ms/step - accuracy: 0.9959 - loss: 0.0199 - val_accuracy: 0.9750 - val_loss: 0.1122
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9620 - loss: 0.1750
RNN Test Accuracy: 0.9620000123977661


Step 10: Train RNN Model

In [ ]:
model_lstm = Sequential([
    Embedding(VOCAB_SIZE, 64, input_length=MAX_LEN),
    LSTM(64),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Step 11: Build LSTM Model

In [ ]:
history_lstm = model_lstm.fit(X_train, y_train, epochs=5, batch_size=32, validation_split=0.2, verbose=1)
loss_lstm, acc_lstm = model_lstm.evaluate(X_test, y_test)
print("LSTM Test Accuracy:", acc_lstm)

Epoch 1/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 11s 77ms/step - accuracy: 0.8537 - loss: 0.3724 - val_accuracy: 0.9312 - val_loss: 0.2579
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 58ms/step - accuracy: 0.9416 - loss: 0.2100 - val_accuracy: 0.9450 - val_loss: 0.1968
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 12s 79ms/step - accuracy: 0.9681 - loss: 0.1462 - val_accuracy: 0.9600 - val_loss: 0.1564
Epoch 4/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 8s 56ms/step - accuracy: 0.9809 - loss: 0.1087 - val_accuracy: 0.9800 - val_loss: 0.0875
Epoch 5/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 8s 80ms/step - accuracy: 0.9834 - loss: 0.0873 - val_accuracy: 0.9712 - val_loss: 0.1016
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9590 - loss: 0.1421
LSTM Test Accuracy: 0.9589999914169312


Step 12: Train the LSTM Model

In [ ]:
# Train LSTM (5 epochs)
history_lstm = model_lstm.fit(X_train, y_train, epochs=5, batch_size=32, validation_split=0.2, verbose=1)

# Evaluate
loss, acc = model_lstm.evaluate(X_test, y_test)
print(f"\n✅ LSTM Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")

Epoch 1/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 8s 80ms/step - accuracy: 0.9859 - loss: 0.0696 - val_accuracy: 0.9812 - val_loss: 0.1301
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 8s 56ms/step - accuracy: 0.9891 - loss: 0.0692 - val_accuracy: 0.9762 - val_loss: 0.1229
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 11s 108ms/step - accuracy: 0.9906 - loss: 0.0527 - val_accuracy: 0.9775 - val_loss: 0.1138
Epoch 4/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 15s 56ms/step - accuracy: 0.9916 - loss: 0.0405 - val_accuracy: 0.9825 - val_loss: 0.0806
Epoch 5/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 65ms/step - accuracy: 0.9919 - loss: 0.0415 - val_accuracy: 0.9812 - val_loss: 0.1284
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9780 - loss: 0.1494

✅ LSTM Test Accuracy: 0.9780 (97.80%)


Step 13: Compare Both Models

In [ ]:
print("Final Results")
print("RNN Test Accuracy:", acc_rnn)
print("LSTM Test Accuracy:", acc_lstm)

Final Results
RNN Test Accuracy: 0.9620000123977661
LSTM Test Accuracy: 0.9589999914169312


Step 14:

In [ ]:
# Combined recovery + Word2Vec cell (run after restart)
!pip install gensim

import gensim.downloader as api
import numpy as np
import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Reload data (use your file path)
file_path = "/content/drive/MyDrive/ai&ml_assessment/10.True vs. Fake News Dataset-20260508T165314Z-3-001/10.True vs. Fake News Dataset/truevsfakenews.csv"
df = pd.read_csv(file_path)
df['label_num'] = df['label'].map({'true':1, 'fake':0})

# Subset
true_df = df[df['label_num']==1]
fake_df = df[df['label_num']==0]
sample_true = true_df.sample(2500, random_state=42)
sample_fake = fake_df.sample(2500, random_state=42)
df_sample = pd.concat([sample_true, sample_fake]).sample(frac=1, random_state=42).reset_index(drop=True)

# Clean text
def clean_text(t):
    t = t.lower()
    t = t.translate(str.maketrans('','', string.punctuation))
    t = re.sub(r'\d+','', t)
    t = re.sub(r'\s+',' ', t).strip()
    return t
df_sample['clean_text'] = df_sample['text'].apply(clean_text)

# Tokenizer
VOCAB_SIZE = 10000
MAX_LEN = 100
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(df_sample['clean_text'])
X = pad_sequences(tokenizer.texts_to_sequences(df_sample['clean_text']), maxlen=MAX_LEN, padding='post')
y = df_sample['label_num'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Word2Vec part
print("Downloading GloVe (50d)...")
embedding_model = api.load('glove-wiki-gigaword-50')
embedding_dim = 50

word_index = tokenizer.word_index
vocab_size = min(VOCAB_SIZE, len(word_index) + 1)
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in word_index.items():
    if i < vocab_size:
        try:
            embedding_matrix[i] = embedding_model[word]
        except KeyError:
            pass

model_w2v = Sequential([
    Embedding(vocab_size, embedding_dim, weights=[embedding_matrix],
              input_length=MAX_LEN, trainable=False),
    LSTM(64, return_sequences=False),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])
model_w2v.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_w2v.summary()

history_w2v = model_w2v.fit(X_train, y_train, epochs=3, batch_size=32,
                            validation_split=0.2, verbose=1)
loss_w2v, acc_w2v = model_w2v.evaluate(X_test, y_test)
print(f"Word2Vec+LSTM Test Accuracy: {acc_w2v:.4f} ({acc_w2v*100:.2f}%)")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 500,000 (1.91 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 500,000 (1.91 MB)

Epoch 1/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.8906 - loss: 0.3053 - val_accuracy: 0.9488 - val_loss: 0.1555
Epoch 2/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.9528 - loss: 0.1489 - val_accuracy: 0.9425 - val_loss: 0.1642
Epoch 3/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.9522 - loss: 0.1434 - val_accuracy: 0.9538 - val_loss: 0.1171
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9570 - loss: 0.1137
Word2Vec+LSTM Test Accuracy: 0.9570 (95.70%)


In [ ]:
!pip install gensim